# Première expérience — HistGradientBoostingRegressor

## Étape 1 — Charger et contrôler les données

**Objectif :** prédire la proportion de oui (`yes_share`, entre 0 et 1) pour un objet de votation dans une commune.

Cette expérience utilise la table structurée existante : **analyse rétrospective sur la géographie 2026**, sans revendication de prévision historique stricte. Les limites de disponibilité des sources sont décrites dans `data/processed/modeling/README.md`.

Cette étape vérifie le contrat des données et identifie les variables candidates.

In [1]:
import os
os.environ["OMP_NUM_THREADS"] = "1"

from pathlib import Path
import hashlib
import json

import numpy as np
import pandas as pd
from IPython.display import display

# Retrouver la racine depuis le dossier de lancement du notebook.
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "environment.yml").exists())
DATA_DIR = ROOT / "data/processed/modeling"
DATA_PATH = DATA_DIR / "municipal_structured.parquet"
manifest = json.loads((DATA_DIR / "manifest.json").read_text(encoding="utf-8"))
data = pd.read_parquet(DATA_PATH)
print("Source :", DATA_PATH.relative_to(ROOT))
print("SHA-256 :", hashlib.sha256(DATA_PATH.read_bytes()).hexdigest())
print("Mode :", manifest["mode"])

Source : data\processed\modeling\municipal_structured.parquet
SHA-256 : c8a488826093a53f307af0b4a04627fa6d78118a8ac4b25ca8c8e097fdf8e043
Mode : exploratory_retrospective


In [2]:
# La liste explicite empêche l'ajout accidentel de résultats ou d'identifiants.
TARGET = manifest["target"]
numeric_features = manifest["numeric_features"]
categorical_features = manifest["categorical_features"]
indicator_features = manifest["missing_indicators"]
feature_columns = numeric_features + categorical_features + indicator_features
required = ["municipality_id", "vote_id", "vote_date", TARGET] + feature_columns
assert not (set(required) - set(data.columns)), "Colonnes requises absentes"
assert len(feature_columns) == len(set(feature_columns)), "Prédicteurs dupliqués"
assert TARGET not in feature_columns
assert not set(feature_columns) & {"municipality_id", "vote_id", "vote_key", "vote_date"}
assert data[["municipality_id", "vote_id", "vote_date"]].notna().all().all()
assert not data.duplicated(["municipality_id", "vote_id"]).any()
assert data.groupby("vote_id")["vote_date"].nunique().eq(1).all()
assert data[TARGET].notna().all() and data[TARGET].between(0, 1).all()
assert np.isfinite(data[TARGET].to_numpy(dtype=float)).all()
assert len(data) == manifest["rows_after_joins"]
assert data["vote_id"].nunique() == manifest["objects"]
assert data["vote_date"].nunique() == manifest["dates"]

summary = pd.Series({
    "Lignes": len(data),
    "Communes": data["municipality_id"].nunique(),
    "Objets": data["vote_id"].nunique(),
    "Dates": data["vote_date"].nunique(),
    "Première date": data["vote_date"].min().date(),
    "Dernière date": data["vote_date"].max().date(),
    "Variables numériques candidates": len(numeric_features),
    "Variables catégorielles candidates": len(categorical_features),
    "Indicateurs de valeurs manquantes": len(indicator_features),
}, name="Valeur")
display(summary.to_frame())
print("Contrôles de structure et de cible réussis.")

Contrôles de structure et de cible réussis.


,Valeur
Lignes,262981
Communes,2105
Objets,125
Dates,44
Première date,2012-03-11
Dernière date,2025-09-28
Variables numériques candidates,19
Variables catégorielles candidates,6
Indicateurs de valeurs manquantes,3


In [3]:
# Couverture annuelle pour préparer un découpage par dates entières.
coverage = (
    data.assign(year=data["vote_date"].dt.year)
    .groupby("year")
    .agg(rows=("vote_id", "size"), objects=("vote_id", "nunique"),
         dates=("vote_date", "nunique"), municipalities=("municipality_id", "nunique"))
)
display(coverage)

feature_audit = pd.DataFrame({
    "dtype": data[feature_columns].dtypes.astype(str),
    "missing_rows": data[feature_columns].isna().sum(),
    "missing_pct": data[feature_columns].isna().mean().mul(100).round(2),
    "unique_values": data[feature_columns].nunique(),
})
feature_audit["role"] = [
    "numeric" if c in numeric_features else "categorical" if c in categorical_features else "indicator"
    for c in feature_audit.index
]
display(feature_audit.sort_values("missing_pct", ascending=False))

,rows,objects,dates,municipalities
year,,,,
2012,25236,12,4,2103
2013,23133,11,4,2103
2014,25236,12,4,2103
2015,12618,6,2,2103
2016,27339,13,4,2103
2017,14721,7,3,2103
2018,21040,10,4,2104
2019,6312,3,2,2104
2020,18936,9,3,2104


,dtype,missing_rows,missing_pct,unique_values,role
party_share_centre,Float64,231511,88.03,2096,numeric
object_domain_3,string,199849,75.99,10,categorical
party_share_pbd,Float64,75046,28.54,5016,numeric
object_domain_2,string,63108,24.00,12,categorical
party_share_pev,Float64,56983,21.67,6420,numeric
party_share_pdc,Float64,36294,13.80,6129,numeric
party_share_pvl,Float64,30931,11.76,7413,numeric
party_share_verts,Float64,10525,4.00,7981,numeric
party_share_ps,Float64,4865,1.85,8170,numeric
party_share_plr,Float64,3913,1.49,8210,numeric


## Étape 2 — Partitions chronologiques et historique figé

Cette étape fixe les ensembles entraînement, validation et test **chronologiquement, par dates entières** : toutes les communes et tous les objets d'une même date restent ensemble. Le test sera réservé à l'évaluation finale.

Les variables `history_mean_yes`, `history_n_objects` et `history_missing` sont candidates : leur historique devra être recalculé selon le protocole choisi (figé ou roulant), avant l'entraînement. La table contient actuellement un historique roulant sur les dates antérieures.

Les bornes sont choisies sur la couverture, avant tout calcul de performance :

- **Entraînement : 2012–2020**, pour apprendre sur la période la plus ancienne.
- **Validation : 2021–2022**, pour comparer les choix du modèle.
- **Test : 2023–2025**, réservé à une évaluation finale après fixation des choix.

Chaque date entière appartient à une seule partition. L'historique d'une ligne d'entraînement utilise uniquement les dates précédentes, y compris avant 2012. Pour la validation et le test, il est **figé à fin 2020** : aucun résultat de validation ou de test n'alimente les prédicteurs. Ce protocole n'inclut pas de réentraînement sur train + validation. Les dates de publication historiques restent non vérifiées : le cadre demeure rétrospectif.

In [4]:
TRAIN_END = pd.Timestamp("2020-12-31")
VALIDATION_END = pd.Timestamp("2022-12-31")
SEED = 42

experiment_data = data.copy()
experiment_data["partition"] = np.select(
    [experiment_data.vote_date.le(TRAIN_END), experiment_data.vote_date.le(VALIDATION_END)],
    ["train", "validation"], default="test",
)
partition_order = ["train", "validation", "test"]
partition_summary = experiment_data.groupby("partition").agg(
    first_date=("vote_date", "min"), last_date=("vote_date", "max"),
    rows=("vote_id", "size"), objects=("vote_id", "nunique"), dates=("vote_date", "nunique"),
).reindex(partition_order)
assert partition_summary.rows.gt(0).all()
assert experiment_data.groupby("vote_date").partition.nunique().eq(1).all()
assert experiment_data.groupby("vote_id").partition.nunique().eq(1).all()
assert partition_summary.loc["train", "last_date"] < partition_summary.loc["validation", "first_date"]
assert partition_summary.loc["validation", "last_date"] < partition_summary.loc["test", "first_date"]
display(partition_summary)

,first_date,last_date,rows,objects,dates
partition,,,,,
train,2012-03-11,2020-11-29,174571,83,30
validation,2021-03-07,2022-09-25,50520,24,7
test,2023-06-18,2025-09-28,37890,18,7


In [5]:
import sys

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from scripts.integration.build_modeling_dataset import clean_outcomes

# Réutiliser exactement les exclusions du constructeur de la table.
history_path = ROOT / "data/processed/ofs-votes/votes_communes.parquet"
history_sha256 = hashlib.sha256(history_path.read_bytes()).hexdigest()
assert history_sha256 == manifest["sources"]["ofs"]["sha256"], "Source OFS modifiée : reconstruire la table"
history_source, _ = clean_outcomes(pd.read_parquet(history_path))

def frozen_history(targets, outcomes, train_end):
    # Seuls les résultats de la période autorisée peuvent entrer dans l'historique.
    pool = outcomes.loc[outcomes.vote_date.le(train_end)]
    result = pd.DataFrame(index=targets.index, columns=[
        "history_mean_yes", "history_n_objects", "history_last_vote_date"
    ])
    for date, group in targets.groupby("vote_date", sort=True):
        past = pool.loc[pool.vote_date.lt(date)]
        stats = past.groupby("municipality_id").agg(
            history_mean_yes=("yes_share", "mean"),
            history_n_objects=("yes_share", "count"),
            history_last_vote_date=("vote_date", "max"),
        )
        for column in result.columns:
            result.loc[group.index, column] = group.municipality_id.map(stats[column]).to_numpy()
    result["history_mean_yes"] = pd.to_numeric(result.history_mean_yes)
    result["history_n_objects"] = pd.to_numeric(result.history_n_objects).fillna(0).astype("int64")
    result["history_last_vote_date"] = pd.to_datetime(result.history_last_vote_date)
    result["history_missing"] = result.history_mean_yes.isna()
    return result

history = frozen_history(experiment_data, history_source, TRAIN_END)
experiment_data[history.columns] = history
known = experiment_data.history_last_vote_date.notna()
assert (experiment_data.loc[known, "history_last_vote_date"] < experiment_data.loc[known, "vote_date"]).all()
assert experiment_data.loc[known, "history_last_vote_date"].le(TRAIN_END).all()

# L'historique d'entraînement doit retrouver celui de la table initiale.
train_mask = experiment_data.partition.eq("train")
np.testing.assert_allclose(
    experiment_data.loc[train_mask, "history_mean_yes"].to_numpy(dtype=float, na_value=np.nan),
    data.loc[train_mask, "history_mean_yes"].to_numpy(dtype=float, na_value=np.nan), equal_nan=True,
)
display(experiment_data.groupby("partition").agg(
    missing_history=("history_missing", "sum"),
    last_history_date=("history_last_vote_date", "max"),
).reindex(partition_order))

,missing_history,last_history_date
partition,,
train,2,2020-09-27
validation,24,2020-11-29
test,18,2020-11-29


In [6]:
# Petit contrôle indépendant : même jour exclu, résultats réservés sans effet.
toy_outcomes = pd.DataFrame({
    "municipality_id": ["a", "a", "a", "a"],
    "vote_date": pd.to_datetime(["2019-01-01", "2020-01-01", "2020-01-01", "2021-01-01"]),
    "yes_share": [0.2, 0.4, 0.8, 1.0],
})
toy_targets = pd.DataFrame({
    "municipality_id": ["a", "a", "a", "new"],
    "vote_date": pd.to_datetime(["2020-01-01", "2021-01-01", "2023-01-01", "2023-01-01"]),
})
toy_history = frozen_history(toy_targets, toy_outcomes, TRAIN_END)
np.testing.assert_allclose(toy_history.history_mean_yes, [0.2, 1.4 / 3, 1.4 / 3, np.nan], equal_nan=True)
assert toy_history.history_n_objects.tolist() == [1, 3, 3, 0]
changed = toy_outcomes.copy()
changed.loc[changed.vote_date.gt(TRAIN_END), "yes_share"] = 0.0
pd.testing.assert_frame_equal(toy_history, frozen_history(toy_targets, changed, TRAIN_END))
print("Contrôles réussis : dates séparées, historique passé et figé, résultats réservés sans effet.")

Contrôles réussis : dates séparées, historique passé et figé, résultats réservés sans effet.


In [7]:
# Préparer les matrices sans ajuster de prétraitement et sans examiner les cibles test.
X_train = experiment_data.loc[experiment_data.partition.eq("train"), feature_columns].copy()
y_train = experiment_data.loc[X_train.index, TARGET].copy()
X_validation = experiment_data.loc[experiment_data.partition.eq("validation"), feature_columns].copy()
y_validation = experiment_data.loc[X_validation.index, TARGET].copy()
test_index = experiment_data.index[experiment_data.partition.eq("test")]

protocol = {
    "model": "HistGradientBoostingRegressor",
    "mode": "exploratory_retrospective",
    "seed": SEED,
    "dataset_sha256": hashlib.sha256(DATA_PATH.read_bytes()).hexdigest(),
    "history_source_sha256": history_sha256,
    "train_end": str(TRAIN_END.date()),
    "validation_end": str(VALIDATION_END.date()),
    "history_policy": "strictly earlier dates; frozen at train_end; pre-2012 outcomes included",
    "refit_train_validation": False,
    "metric": "MAE in percentage points, equal weight per municipality-object row",
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "indicator_features": indicator_features,
    "partition_dates": {
        part: sorted(experiment_data.loc[experiment_data.partition.eq(part), "vote_date"].dt.strftime("%Y-%m-%d").unique().tolist())
        for part in partition_order
    },
}
output_dir = ROOT / "reports/generated/hist_gradient_boosting"
output_dir.mkdir(parents=True, exist_ok=True)
(output_dir / "protocol.json").write_text(json.dumps(protocol, ensure_ascii=False, indent=2), encoding="utf-8")
partition_summary.to_csv(output_dir / "partitions.csv")
print("Train :", X_train.shape, "; validation :", X_validation.shape, "; lignes test réservées :", len(test_index))
print("Protocole enregistré dans", output_dir.relative_to(ROOT))

Train : (174571, 28) ; validation : (50520, 28) ; lignes test réservées : 37890
Protocole enregistré dans reports\generated\hist_gradient_boosting


La prochaine étape consiste à entraîner les baselines et un premier `HistGradientBoostingRegressor`, puis à comparer leurs MAE sur la validation uniquement. Le test reste réservé.

## Étape 3 — Premier entraînement et comparaison sur la validation

Trois prédictions sont comparées sur exactement les mêmes lignes :

- **Constante** : moyenne de `yes_share` sur les lignes d'entraînement.
- **Historique communal** : moyenne historique figée fin 2020 ; repli sur la constante pour une commune sans historique.
- **HistGradientBoostingRegressor** : les 28 prédicteurs retenus, avec paramètres fixés avant de mesurer les scores.

Premier réglage : perte quadratique, 100 itérations, taux d'apprentissage 0,1, au plus 31 feuilles et au moins 20 observations par feuille. L'arrêt anticipé est désactivé pour éviter une séparation interne aléatoire. Aucune recherche d'hyperparamètres n'est effectuée.

Les catégories sont encodées sur l'entraînement uniquement et déclarées catégorielles au modèle. Les catégories inconnues deviennent manquantes. Le modèle gère directement les valeurs manquantes ; aucune normalisation ni imputation n'est appliquée. Les prédictions sont bornées entre 0 et 1.

La MAE principale donne le même poids à chaque ligne commune × objet. Les MAE moyennes par objet et par date sont aussi présentées pour éclairer cette pondération. Tous les scores portent sur **2021–2022**, jamais sur le test.

In [8]:
from time import perf_counter
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingRegressor

def model_input(frame):
    # Harmoniser les types pandas sans apprendre de statistique sur les données.
    result = frame.copy()
    for column in numeric_features + indicator_features:
        result[column] = result[column].astype(float)
    for column in categorical_features:
        result[column] = result[column].astype(object).where(result[column].notna(), np.nan)
    return result

train_input = model_input(X_train)
validation_input = model_input(X_validation)
assert train_input[categorical_features].nunique().le(255).all()
empty_training_features = train_input.columns[train_input.isna().all()].tolist()
continuous_columns = [c for c in numeric_features + indicator_features if c not in empty_training_features]
active_categorical = [c for c in categorical_features if c not in empty_training_features]
print("Excluded all-missing training columns:", empty_training_features)
preprocessor = ColumnTransformer([
    ("numeric", "passthrough", continuous_columns),
    ("categorical", OrdinalEncoder(
        handle_unknown="use_encoded_value", unknown_value=np.nan,
        encoded_missing_value=np.nan, dtype=float,
    ), active_categorical),
], sparse_threshold=0)
settings = dict(
    loss="squared_error", learning_rate=0.1, max_iter=100,
    max_leaf_nodes=31, min_samples_leaf=20, l2_regularization=0.0,
    early_stopping=False, random_state=SEED,
)
model = Pipeline([
    ("prepare", preprocessor),
    ("regressor", HistGradientBoostingRegressor(
        **settings,
        categorical_features=[False] * len(continuous_columns) + [True] * len(active_categorical),
    )),
])
started = perf_counter()
model.fit(train_input, y_train.astype(float))
training_seconds = perf_counter() - started
raw_prediction = model.predict(validation_input)
assert np.isfinite(raw_prediction).all()
print("scikit-learn :", sklearn.__version__)
print("Itérations :", model.named_steps["regressor"].n_iter_)
print(f"Durée d'entraînement : {training_seconds:.2f} s")

Excluded all-missing training columns: ['party_share_centre']
scikit-learn : 1.9.1
Itérations : 100
Durée d'entraînement : 3.63 s


In [9]:
constant_prediction = float(y_train.mean())
predictions = {
    "constant": np.full(len(y_validation), constant_prediction),
    "municipal_history": X_validation.history_mean_yes.fillna(constant_prediction).to_numpy(dtype=float),
    "hist_gradient_boosting": np.clip(raw_prediction, 0, 1),
}
validation_predictions = experiment_data.loc[
    X_validation.index, ["municipality_id", "vote_id", "vote_date", TARGET]
].copy()
score_rows = []
object_tables = []
date_tables = []
for name, prediction in predictions.items():
    assert len(prediction) == len(y_validation) and np.isfinite(prediction).all()
    assert ((prediction >= 0) & (prediction <= 1)).all()
    validation_predictions[name] = prediction
    errors = validation_predictions[["vote_id", "vote_date"]].copy()
    errors["absolute_error_pp"] = np.abs(y_validation.to_numpy(dtype=float) - prediction) * 100
    by_object = errors.groupby(["vote_id", "vote_date"], as_index=False).absolute_error_pp.mean()
    by_date = errors.groupby("vote_date", as_index=False).absolute_error_pp.mean()
    object_tables.append(by_object.assign(model=name))
    date_tables.append(by_date.assign(model=name))
    score_rows.append({
        "model": name,
        "mae_pp": errors.absolute_error_pp.mean(),
        "mean_object_mae_pp": by_object.absolute_error_pp.mean(),
        "mean_date_mae_pp": by_date.absolute_error_pp.mean(),
    })
scores = pd.DataFrame(score_rows).set_index("model")
scores["gain_vs_constant_pp"] = scores.loc["constant", "mae_pp"] - scores.mae_pp
scores["gain_vs_history_pp"] = scores.loc["municipal_history", "mae_pp"] - scores.mae_pp
display(scores.round(3))
object_scores = pd.concat(object_tables, ignore_index=True)
date_scores = pd.concat(date_tables, ignore_index=True)
display(date_scores.pivot(index="vote_date", columns="model", values="absolute_error_pp").round(3))
print("Replis de la baseline historique :", int(X_validation.history_mean_yes.isna().sum()))
print("Prédictions boosting bornées :", int(((raw_prediction < 0) | (raw_prediction > 1)).sum()))

Replis de la baseline historique : 24
Prédictions boosting bornées : 0


,mae_pp,mean_object_mae_pp,mean_date_mae_pp,gain_vs_constant_pp,gain_vs_history_pp
model,,,,,
constant,13.173,13.173,13.388,0.000,-0.203
municipal_history,12.970,12.970,13.143,0.203,0.000
hist_gradient_boosting,12.005,12.005,12.345,1.167,0.965


model,constant,hist_gradient_boosting,municipal_history
vote_date,,,
2021-03-07,10.571,17.133,11.000
2021-06-13,12.784,8.612,12.498
2021-09-26,14.961,11.526,14.813
2021-11-28,13.856,12.719,13.220
2022-02-13,13.767,10.602,13.973
2022-05-15,17.166,13.242,15.364
2022-09-25,10.613,12.582,11.129


In [10]:
import platform
import joblib

run_manifest = {
    **protocol,
    "stage": "first_validation_run",
    "thread_limit": 1,
    "excluded_all_missing_training_features": empty_training_features,
    "effective_features": continuous_columns + active_categorical,
    "settings": settings,
    "iterations": int(model.named_steps["regressor"].n_iter_),
    "training_seconds": training_seconds,
    "training_rows": len(X_train),
    "validation_rows": len(X_validation),
    "validation_objects": validation_predictions.vote_id.nunique(),
    "validation_dates": validation_predictions.vote_date.nunique(),
    "constant_prediction": constant_prediction,
    "prediction_clipping": [0, 1],
    "clipped_validation_predictions": int(((raw_prediction < 0) | (raw_prediction > 1)).sum()),
    "preprocessing": "training-only ordinal encoding, native categorical splits and missing values; no imputation or scaling",
    "versions": {"python": platform.python_version(), "sklearn": sklearn.__version__, "pandas": pd.__version__, "numpy": np.__version__},
    "test_evaluated": False,
}
scores.to_csv(output_dir / "validation_metrics.csv")
validation_predictions.to_parquet(output_dir / "validation_predictions.parquet", index=False)
object_scores.to_csv(output_dir / "validation_by_object.csv", index=False)
date_scores.to_csv(output_dir / "validation_by_date.csv", index=False)
(output_dir / "run_manifest.json").write_text(json.dumps(run_manifest, ensure_ascii=False, indent=2), encoding="utf-8")
model_dir = ROOT / "models/hist_gradient_boosting"
model_dir.mkdir(parents=True, exist_ok=True)
joblib.dump({"pipeline": model, "features": feature_columns, "manifest": run_manifest}, model_dir / "first_validation.joblib")
print("Résultats :", output_dir.relative_to(ROOT))
print("Modèle :", (model_dir / "first_validation.joblib").relative_to(ROOT))
print("Test réservé : aucun score calculé.")

Résultats : reports\generated\hist_gradient_boosting
Modèle : models\hist_gradient_boosting\first_validation.joblib
Test réservé : aucun score calculé.


**Lecture des résultats :** une MAE de 10 points signifie un écart absolu moyen de 10 points de pourcentage sur la proportion de oui. Un gain positif indique une amélioration sur la baseline correspondante.

Il s'agit d'une première mesure de validation rétrospective, avec seulement 24 objets sur 7 dates. Les nombreuses communes d'un objet ne constituent pas autant de votations indépendantes. Les résultats ne démontrent donc pas, à eux seuls, une capacité de prévision historique stricte ou de généralisation sur le test.

### Premier résultat observé

| Modèle | MAE validation (points de pourcentage) |
|---|---:|
| Constante | 13,173 |
| Historique communal | 12,970 |
| HistGradientBoostingRegressor | 12,005 |

Le boosting gagne **0,965 point** sur l'historique communal (environ **7,4 %** de baisse de MAE) et **1,167 point** sur la constante. Ces scores portent sur les 50 520 lignes de validation, 24 objets et 7 dates.

Les 100 itérations ont été ajustées sur 174 571 lignes de 2012–2020. Parmi les 28 variables candidates, `party_share_centre` est entièrement absente de l'entraînement : elle est exclue avant ajustement, ce qui laisse **27 prédicteurs effectifs**. Ce filtre utilise uniquement l'entraînement. Les autres valeurs manquantes restent gérées par le modèle.

Exécuter le notebook depuis un noyau neuf : `OMP_NUM_THREADS=1` est fixé avant les imports pour ce premier essai sous Windows. Les cellules ont été exécutées dans IPython avec l'environnement `swiss-votes-ml` ; les sorties sont conservées. Le test 2023–2025 reste non évalué.

## Étape 4 — Écart entraînement/validation et erreurs par objet

Cette section recharge le modèle sauvegardé de l'étape 3, sans le réentraîner. Elle mesure son erreur sur les données ayant servi à l'ajustement, puis détaille les erreurs de validation. Les prédictions test ne sont pas calculées.

Un écart important entre entraînement et validation est un signal de surapprentissage possible, mais aussi de différences entre périodes et objets. Il ne permet pas à lui seul de séparer ces causes. L'erreur d'entraînement est une mesure optimiste, pas une nouvelle estimation de généralisation.

In [11]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
from pathlib import Path
import hashlib
import json
import joblib
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "environment.yml").exists())
diagnostic_dir = ROOT / "reports/generated/hist_gradient_boosting"
saved = joblib.load(ROOT / "models/hist_gradient_boosting/first_validation.joblib")
saved_manifest = saved["manifest"]
source_path = ROOT / "data/processed/modeling/municipal_structured.parquet"
assert hashlib.sha256(source_path.read_bytes()).hexdigest() == saved_manifest["dataset_sha256"]
diagnostic_data = pd.read_parquet(source_path)
training = diagnostic_data.loc[diagnostic_data.vote_date.le(pd.Timestamp(saved_manifest["train_end"]))].copy()
assert len(training) == saved_manifest["training_rows"]

# L'historique initial est identique à l'historique recalculé sur le train (contrôle étape 2).
training_input = training[saved["features"]].copy()
for column in saved_manifest["numeric_features"] + saved_manifest["indicator_features"]:
    training_input[column] = training_input[column].astype(float)
for column in saved_manifest["categorical_features"]:
    training_input[column] = training_input[column].astype(object).where(training_input[column].notna(), np.nan)
training_predictions = {
    "constant": np.full(len(training), saved_manifest["constant_prediction"]),
    "municipal_history": training.history_mean_yes.fillna(saved_manifest["constant_prediction"]).to_numpy(dtype=float),
    "hist_gradient_boosting": np.clip(saved["pipeline"].predict(training_input), 0, 1),
}
# La constante de repli est celle du modèle de comparaison, ajustée sur tout le train.
training_mae = pd.Series({
    name: np.abs(training.yes_share.to_numpy(dtype=float) - values).mean() * 100
    for name, values in training_predictions.items()
}, name="train_mae_pp")
validation_metrics = pd.read_csv(diagnostic_dir / "validation_metrics.csv", index_col="model")
generalization = training_mae.to_frame().join(validation_metrics.mae_pp.rename("validation_mae_pp"))
generalization["gap_pp"] = generalization.validation_mae_pp - generalization.train_mae_pp
display(generalization.round(3))
generalization.to_csv(diagnostic_dir / "train_validation_gap.csv")

,train_mae_pp,validation_mae_pp,gap_pp
constant,16.424,13.173,-3.251
municipal_history,16.554,12.970,-3.584
hist_gradient_boosting,4.921,12.005,7.084


In [12]:
validation_detail = pd.read_parquet(diagnostic_dir / "validation_predictions.parquet")
assert len(validation_detail) == saved_manifest["validation_rows"]
assert set(validation_detail.vote_date.dt.strftime("%Y-%m-%d")) == set(saved_manifest["partition_dates"]["validation"])
assert validation_detail.vote_date.max() <= pd.Timestamp(saved_manifest["validation_end"])
validation_detail["vote_id"] = validation_detail.vote_id.astype("string")
validation_detail["boosting_error_pp"] = (validation_detail.hist_gradient_boosting - validation_detail.yes_share) * 100
validation_detail["boosting_abs_pp"] = validation_detail.boosting_error_pp.abs()
validation_detail["history_abs_pp"] = (validation_detail.municipal_history - validation_detail.yes_share).abs() * 100
validation_detail["constant_abs_pp"] = (validation_detail.constant - validation_detail.yes_share).abs() * 100
object_diagnostics = validation_detail.groupby(["vote_id", "vote_date"], as_index=False).agg(
    rows=("municipality_id", "size"),
    boosting_mae_pp=("boosting_abs_pp", "mean"),
    history_mae_pp=("history_abs_pp", "mean"),
    constant_mae_pp=("constant_abs_pp", "mean"),
    bias_pp=("boosting_error_pp", "mean"),
    observed_municipal_mean=("yes_share", "mean"),
    predicted_municipal_mean=("hist_gradient_boosting", "mean"),
)
object_diagnostics["gain_vs_history_pp"] = object_diagnostics.history_mae_pp - object_diagnostics.boosting_mae_pp

# Les titres servent uniquement à lire les résultats, jamais à entraîner le modèle.
swissvotes = pd.read_parquet(ROOT / "data/processed/swissvotes/swissvotes_clean.parquet")
titles = swissvotes[["vote_key", "titel_kurz_f"]].rename(columns={"vote_key": "vote_id", "titel_kurz_f": "title"})
titles["vote_id"] = titles.vote_id.astype("string")
assert not titles.vote_id.duplicated().any()
object_diagnostics = object_diagnostics.merge(titles, on="vote_id", how="left", validate="many_to_one")
assert object_diagnostics.title.notna().all()
object_diagnostics = object_diagnostics.sort_values("boosting_mae_pp", ascending=False)
columns = ["vote_id", "vote_date", "title", "boosting_mae_pp", "history_mae_pp", "gain_vs_history_pp", "bias_pp"]
print("Objets les plus difficiles (MAE boosting décroissante) :")
display(object_diagnostics[columns].round(3))
print("Gains les plus importants sur l'historique :")
display(object_diagnostics.nlargest(5, "gain_vs_history_pp")[columns].round(3))
print("Dégradations les plus importantes par rapport à l'historique :")
display(object_diagnostics.nsmallest(5, "gain_vs_history_pp")[columns].round(3))
object_diagnostics.to_csv(diagnostic_dir / "validation_object_diagnostics.csv", index=False)

date_diagnostics = validation_detail.groupby("vote_date").agg(
    boosting_mae_pp=("boosting_abs_pp", "mean"), history_mae_pp=("history_abs_pp", "mean"),
)
date_diagnostics["gain_vs_history_pp"] = date_diagnostics.history_mae_pp - date_diagnostics.boosting_mae_pp
diagnostic_summary = {
    "objects_improved_vs_history": int(object_diagnostics.gain_vs_history_pp.gt(0).sum()),
    "objects_total": len(object_diagnostics),
    "dates_improved_vs_history": int(date_diagnostics.gain_vs_history_pp.gt(0).sum()),
    "dates_total": len(date_diagnostics),
    "model_refitted": False,
    "test_evaluated": False,
}
display(date_diagnostics.round(3))
print(json.dumps(diagnostic_summary, indent=2))
(diagnostic_dir / "diagnostic_summary.json").write_text(json.dumps(diagnostic_summary, indent=2), encoding="utf-8")

Objets les plus difficiles (MAE boosting décroissante) :
Gains les plus importants sur l'historique :
Dégradations les plus importantes par rapport à l'historique :
{
  "objects_improved_vs_history": 13,
  "objects_total": 24,
  "dates_improved_vs_history": 5,
  "dates_total": 7,
  "model_refitted": false,
  "test_evaluated": false
}
Out[0]: 170


<ipython-input-1-4a13a4e192d6>:31: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  display(object_diagnostics[columns].round(3))
<ipython-input-1-4a13a4e192d6>:33: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  display(object_diagnostics.nlargest(5, "gain_vs_history_pp")[columns].round(3))
<ipython-input-1-4a13a4e192d6>:35: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  display(object_diagnostics.nsmallest(5, "gain_vs_history_pp")[columns].round(3))


,vote_id,vote_date,title,boosting_mae_pp,history_mae_pp,gain_vs_history_pp,bias_pp
19,6570,2022-05-15,Contribution au corps européen de garde-fronti...,26.849,23.842,-3.007,-26.799
10,6480,2021-11-28,Initiative sur les soins infirmiers,22.395,11.782,-10.613,-22.393
0,6380,2021-03-07,Initiative pour l’interdiction de se dissimule...,22.106,11.082,-11.023,-22.073
9,6470,2021-09-26,Mariage pour tous,17.474,12.406,-5.069,-16.898
1,6390,2021-03-07,Loi sur les services d’identification électron...,16.199,12.497,-3.702,15.815
14,6520,2022-02-13,Initiative «Enfants et jeunes sans publicité p...,15.833,7.499,-8.334,-15.268
22,6600,2022-09-25,Loi sur l’AVS (AVS 21),15.515,9.88,-5.635,-6.285
2,6400,2021-03-07,Accord de libre-échange avec l'Indonésie,13.093,9.422,-3.671,3.352
20,6580,2022-09-25,Initiative sur l’élevage intensif,12.967,18.597,5.631,12.111
13,6510,2022-02-13,Initiative pour l'interdiction de l'expériment...,12.858,27.456,14.598,12.744


,vote_id,vote_date,title,boosting_mae_pp,history_mae_pp,gain_vs_history_pp,bias_pp
13,6510,2022-02-13,Initiative pour l'interdiction de l'expériment...,12.858,27.456,14.598,12.744
8,6460,2021-09-26,«Initiative 99%»,5.578,17.22,11.642,-2.573
11,6490,2021-11-28,Initiative sur la justice,6.117,17.031,10.914,4.399
3,6410,2021-06-13,Initiative pour une eau potable propre,8.344,16.731,8.387,2.919
4,6420,2021-06-13,Initiative sur les pesticides,7.885,15.967,8.082,3.81


,vote_id,vote_date,title,boosting_mae_pp,history_mae_pp,gain_vs_history_pp,bias_pp
0,6380,2021-03-07,Initiative pour l’interdiction de se dissimule...,22.106,11.082,-11.023,-22.073
10,6480,2021-11-28,Initiative sur les soins infirmiers,22.395,11.782,-10.613,-22.393
14,6520,2022-02-13,Initiative «Enfants et jeunes sans publicité p...,15.833,7.499,-8.334,-15.268
22,6600,2022-09-25,Loi sur l’AVS (AVS 21),15.515,9.88,-5.635,-6.285
9,6470,2021-09-26,Mariage pour tous,17.474,12.406,-5.069,-16.898


,boosting_mae_pp,history_mae_pp,gain_vs_history_pp
vote_date,,,
2021-03-07,17.133,11.0,-6.132
2021-06-13,8.612,12.498,3.886
2021-09-26,11.526,14.813,3.287
2021-11-28,12.719,13.22,0.501
2022-02-13,10.602,13.973,3.371
2022-05-15,13.242,15.364,2.122
2022-09-25,12.582,11.129,-1.453


Le biais signé est la moyenne de **prédiction − observation** : positif = surestimation du oui, négatif = sous-estimation. Les moyennes observées et prédites sont des moyennes communales non pondérées, pas le résultat national pondéré par les suffrages.

Les résultats par objet aident à localiser les échecs, mais ne prouvent pas leurs causes. Les comparaisons entraînement/validation mêlent différence de période, objets différents et historique figé en validation. Toute modification guidée par ces résultats reste une décision sur validation ; le test doit rester réservé.

### Conclusion du diagnostic

- MAE boosting : **4,921 points sur le train**, **12,005 sur la validation**, soit un écart de **7,084 points**. Le modèle ajuste beaucoup mieux les objets vus que les objets futurs. C'est un signal de surapprentissage possible, sans isoler l'effet du changement de période.
- Le boosting améliore l'historique sur **13 objets sur 24** et **5 dates sur 7**. Le gain moyen cache donc des résultats hétérogènes.
- Gains importants : initiative sur l'expérimentation animale et humaine (**14,60 points**), initiative 99 % (**11,64**) et initiative sur la justice (**10,91**).
- Dégradations importantes : interdiction de se dissimuler le visage (**11,02 points d'erreur en plus**) et soins infirmiers (**10,61**). Le oui est sous-estimé en moyenne de **22,07** et **22,39 points** respectivement.
- Frontex est l'objet le plus difficile pour le boosting : **26,85 points de MAE**, avec une sous-estimation moyenne de **26,80 points**.

Ce premier modèle apporte un gain global modeste, mais sa robustesse n'est pas établie. La suite raisonnable est une comparaison limitée de modèles moins complexes sur la validation (moins de feuilles, davantage d'observations par feuille ou régularisation), sans toucher au test. Ces choix ne sont pas encore exécutés.

Le diagnostic utilise le modèle sauvegardé, sans réentraînement ; ses deux cellules ont été exécutées dans IPython. Les tableaux détaillés sont exportés dans `reports/generated/hist_gradient_boosting/`.

## Étape 5 — Comparaison limitée de modèles moins complexes

Trois candidats sont fixés avant exécution, puis comparés au modèle initial :

| Candidat | Feuilles maximales | Observations minimales par feuille | L2 |
|---|---:|---:|---:|
| Initial | 31 | 20 | 0 |
| Compact | 15 | 100 | 1 |
| Très compact | 7 | 200 | 5 |
| Régularisé | 15 | 200 | 10 |

Chaque nouveau modèle est ajusté depuis zéro sur 2012–2020, avec 100 itérations, un taux d'apprentissage de 0,1 et une graine de 42. L'encodage est réajusté uniquement sur le train. Les sources, les prédicteurs et l'historique figé sont inchangés. Le meilleur candidat, **initial compris**, est choisi sur la MAE de validation 2021–2022 ; le test reste réservé.

Cette comparaison modifie plusieurs paramètres ensemble : elle évalue des configurations, sans isoler l'effet de chaque paramètre. La validation sert maintenant à la sélection ; son meilleur score n'est pas une estimation indépendante de performance finale.

In [13]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
from pathlib import Path
import sys
import hashlib
import json
from time import perf_counter
import joblib
import numpy as np
import pandas as pd
from sklearn.base import clone
from IPython.display import display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "environment.yml").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from scripts.integration.build_modeling_dataset import clean_outcomes

tuning_dir = ROOT / "reports/generated/hist_gradient_boosting/regularization"
tuning_dir.mkdir(parents=True, exist_ok=True)
original = joblib.load(ROOT / "models/hist_gradient_boosting/first_validation.joblib")
original_manifest = original["manifest"]
table_path = ROOT / "data/processed/modeling/municipal_structured.parquet"
assert hashlib.sha256(table_path.read_bytes()).hexdigest() == original_manifest["dataset_sha256"]
tuning_data = pd.read_parquet(table_path)
train_end = pd.Timestamp(original_manifest["train_end"])
validation_end = pd.Timestamp(original_manifest["validation_end"])
tuning_train = tuning_data.loc[tuning_data.vote_date.le(train_end)].copy()
tuning_validation = tuning_data.loc[tuning_data.vote_date.gt(train_end) & tuning_data.vote_date.le(validation_end)].copy()

# Recalculer l'historique de validation avec les seuls résultats autorisés.
ofs_path = ROOT / "data/processed/ofs-votes/votes_communes.parquet"
assert hashlib.sha256(ofs_path.read_bytes()).hexdigest() == original_manifest["history_source_sha256"]
eligible_outcomes, _ = clean_outcomes(pd.read_parquet(ofs_path))
frozen_stats = eligible_outcomes.loc[eligible_outcomes.vote_date.le(train_end)].groupby("municipality_id").agg(
    history_mean_yes=("yes_share", "mean"), history_n_objects=("yes_share", "count"),
)
for column in frozen_stats.columns:
    tuning_validation[column] = tuning_validation.municipality_id.map(frozen_stats[column])
tuning_validation["history_n_objects"] = tuning_validation.history_n_objects.fillna(0).astype("int64")
tuning_validation["history_missing"] = tuning_validation.history_mean_yes.isna()

def tuning_input(frame):
    matrix = frame[original["features"]].copy()
    for column in original_manifest["numeric_features"] + original_manifest["indicator_features"]:
        matrix[column] = matrix[column].astype(float)
    for column in original_manifest["categorical_features"]:
        matrix[column] = matrix[column].astype(object).where(matrix[column].notna(), np.nan)
    return matrix

tuning_X_train = tuning_input(tuning_train)
tuning_X_validation = tuning_input(tuning_validation)
tuning_y_train = tuning_train.yes_share.to_numpy(dtype=float)
tuning_y_validation = tuning_validation.yes_share.to_numpy(dtype=float)
stored_validation = pd.read_parquet(ROOT / "reports/generated/hist_gradient_boosting/validation_predictions.parquet")
assert list(zip(tuning_validation.municipality_id, tuning_validation.vote_id)) == list(zip(stored_validation.municipality_id, stored_validation.vote_id))
np.testing.assert_allclose(tuning_y_validation, stored_validation.yes_share.to_numpy(dtype=float))
np.testing.assert_allclose(
    np.clip(original["pipeline"].predict(tuning_X_validation), 0, 1),
    stored_validation.hist_gradient_boosting.to_numpy(dtype=float), rtol=1e-12, atol=1e-12,
)
print("Même population et mêmes prédictions initiales vérifiées ; aucun résultat test utilisé.")

Même population et mêmes prédictions initiales vérifiées ; aucun résultat test utilisé.


In [14]:
candidates = {
    "compact": {"max_leaf_nodes": 15, "min_samples_leaf": 100, "l2_regularization": 1.0},
    "very_compact": {"max_leaf_nodes": 7, "min_samples_leaf": 200, "l2_regularization": 5.0},
    "regularized": {"max_leaf_nodes": 15, "min_samples_leaf": 200, "l2_regularization": 10.0},
}
# Conserver le plan exact avant d'observer les nouveaux scores.
(tuning_dir / "candidate_plan.json").write_text(json.dumps(candidates, indent=2), encoding="utf-8")
fitted_candidates = {"initial": original["pipeline"]}
candidate_predictions = {}
comparison_rows = []
for name, changes in {"initial": {}, **candidates}.items():
    if name == "initial":
        fitted = original["pipeline"]
        fit_seconds = original_manifest["training_seconds"]
    else:
        fitted = clone(original["pipeline"])
        fitted.set_params(**{f"regressor__{key}": value for key, value in changes.items()})
        started = perf_counter()
        fitted.fit(tuning_X_train, tuning_y_train)
        fit_seconds = perf_counter() - started
        fitted_candidates[name] = fitted
    train_prediction = np.clip(fitted.predict(tuning_X_train), 0, 1)
    validation_prediction = np.clip(fitted.predict(tuning_X_validation), 0, 1)
    assert np.isfinite(train_prediction).all() and np.isfinite(validation_prediction).all()
    candidate_predictions[name] = validation_prediction
    train_mae = float(np.abs(tuning_y_train - train_prediction).mean() * 100)
    validation_mae = float(np.abs(tuning_y_validation - validation_prediction).mean() * 100)
    parameters = fitted.named_steps["regressor"].get_params()
    comparison_rows.append({
        "candidate": name, "train_mae_pp": train_mae,
        "validation_mae_pp": validation_mae, "gap_pp": validation_mae - train_mae,
        "training_seconds": fit_seconds,
        **{key: parameters[key] for key in ["max_leaf_nodes", "min_samples_leaf", "l2_regularization", "max_iter", "learning_rate"]},
    })
comparison = pd.DataFrame(comparison_rows).set_index("candidate")
comparison["gain_vs_initial_pp"] = comparison.loc["initial", "validation_mae_pp"] - comparison.validation_mae_pp
comparison["gain_vs_history_pp"] = float(np.abs(tuning_y_validation - stored_validation.municipal_history.to_numpy(dtype=float)).mean() * 100) - comparison.validation_mae_pp
comparison = comparison.sort_values("validation_mae_pp", kind="stable")
selected_name = comparison.index[0]
display(comparison.round(3))
comparison.to_csv(tuning_dir / "comparison.csv")
print("Configuration retenue sur validation :", selected_name)

Configuration retenue sur validation : very_compact


,train_mae_pp,validation_mae_pp,gap_pp,training_seconds,max_leaf_nodes,min_samples_leaf,l2_regularization,max_iter,learning_rate,gain_vs_initial_pp,gain_vs_history_pp
candidate,,,,,,,,,,,
very_compact,7.601,10.709,3.108,2.381,7,200,5.0,100,0.1,1.297,2.261
compact,6.042,10.944,4.902,2.902,15,100,1.0,100,0.1,1.061,2.026
regularized,6.025,11.239,5.214,2.784,15,200,10.0,100,0.1,0.766,1.731
initial,4.921,12.005,7.084,3.629,31,20,0.0,100,0.1,0.000,0.965


In [15]:
tuning_details = tuning_validation[["municipality_id", "vote_id", "vote_date", "yes_share"]].copy()
for name, prediction in candidate_predictions.items():
    tuning_details[name] = prediction
tuning_details["selected_error_pp"] = np.abs(tuning_details[selected_name] - tuning_details.yes_share) * 100
tuning_details["initial_error_pp"] = np.abs(tuning_details.initial - tuning_details.yes_share) * 100
by_object_tuning = tuning_details.groupby(["vote_id", "vote_date"]).agg(
    selected_mae_pp=("selected_error_pp", "mean"), initial_mae_pp=("initial_error_pp", "mean"),
)
by_date_tuning = tuning_details.groupby("vote_date").agg(
    selected_mae_pp=("selected_error_pp", "mean"), initial_mae_pp=("initial_error_pp", "mean"),
)
for frame in (by_object_tuning, by_date_tuning):
    frame["gain_vs_initial_pp"] = frame.initial_mae_pp - frame.selected_mae_pp
display(by_date_tuning.round(3))
selected_manifest = {
    **original_manifest,
    "stage": "regularization_validation_selection",
    "selected_candidate": selected_name,
    "settings": {**original_manifest["settings"], **candidates.get(selected_name, {})},
    "training_seconds": float(comparison.loc[selected_name, "training_seconds"]),
    "selection_rule": "lowest equal-row validation MAE among initial and three prespecified candidates",
    "validation_mae_pp": float(comparison.loc[selected_name, "validation_mae_pp"]),
    "candidates": candidates,
    "objects_improved_vs_initial": int(by_object_tuning.gain_vs_initial_pp.gt(0).sum()),
    "objects_total": len(by_object_tuning),
    "dates_improved_vs_initial": int(by_date_tuning.gain_vs_initial_pp.gt(0).sum()),
    "dates_total": len(by_date_tuning),
    "test_evaluated": False,
}
# Le comptage de bornage du premier essai ne décrit pas le nouveau modèle.
selected_raw = fitted_candidates[selected_name].predict(tuning_X_validation)
selected_manifest["clipped_validation_predictions"] = int(((selected_raw < 0) | (selected_raw > 1)).sum())
selected_manifest["iterations"] = int(fitted_candidates[selected_name].named_steps["regressor"].n_iter_)
(tuning_dir / "selected_manifest.json").write_text(json.dumps(selected_manifest, ensure_ascii=False, indent=2), encoding="utf-8")
tuning_details.to_parquet(tuning_dir / "validation_predictions.parquet", index=False)
by_object_tuning.to_csv(tuning_dir / "validation_by_object.csv")
by_date_tuning.to_csv(tuning_dir / "validation_by_date.csv")
joblib.dump({"pipeline": fitted_candidates[selected_name], "features": original["features"], "manifest": selected_manifest},
            ROOT / "models/hist_gradient_boosting/selected_regularization.joblib")
print("Objets améliorés :", selected_manifest["objects_improved_vs_initial"], "/", len(by_object_tuning))
print("Dates améliorées :", selected_manifest["dates_improved_vs_initial"], "/", len(by_date_tuning))
print("Modèle initial conservé ; modèle sélectionné sauvegardé séparément. Test non évalué.")

Objets améliorés : 14 / 24
Dates améliorées : 5 / 7
Modèle initial conservé ; modèle sélectionné sauvegardé séparément. Test non évalué.


,selected_mae_pp,initial_mae_pp,gain_vs_initial_pp
vote_date,,,
2021-03-07,18.093,17.133,-0.96
2021-06-13,8.849,8.612,-0.238
2021-09-26,9.548,11.526,1.978
2021-11-28,9.329,12.719,3.39
2022-02-13,8.208,10.602,2.395
2022-05-15,12.588,13.242,0.653
2022-09-25,10.202,12.582,2.381


### Bilan de la comparaison

Le candidat **very_compact** est retenu : 7 feuilles au maximum, 200 observations au minimum par feuille et L2 = 5, avec 100 itérations et un taux d'apprentissage de 0,1.

| Configuration | MAE train | MAE validation | Écart |
|---|---:|---:|---:|
| Initial | 4,921 | 12,005 | 7,084 |
| Compact | 6,042 | 10,944 | 4,902 |
| Très compact | 7,601 | **10,709** | **3,108** |
| Régularisé | 6,025 | 11,239 | 5,214 |

Toutes les valeurs sont des points de pourcentage. Le meilleur candidat gagne **1,297 point** sur le modèle initial (environ **10,8 %** de baisse de MAE) et **2,261 points** sur l'historique communal (environ **17,4 %**). Il améliore le modèle initial sur **14 objets sur 24** et **5 dates sur 7**.

L'erreur d'entraînement augmente tandis que l'erreur de validation diminue : ces résultats soutiennent le choix d'un modèle moins complexe pour cette expérience. L'écart train/validation diminue, sans disparaître. La validation a servi à choisir le modèle ; une confirmation sur le test réservé reste nécessaire avant de conclure sur sa généralisation.

Les trois nouveaux modèles ont été entraînés et les cellules exécutées dans IPython. Le modèle retenu est sauvegardé dans `models/hist_gradient_boosting/selected_regularization.joblib`, et les comparaisons dans `reports/generated/hist_gradient_boosting/regularization/`. Aucun score test n'a été calculé.

## Étape 6 — Évaluation finale sur le test 2023–2025

Les choix sont fixés avant de calculer les scores test : modèle `very_compact`, 7 feuilles maximum, 200 observations minimum par feuille, L2 = 5, 100 itérations, taux d'apprentissage 0,1 et graine 42. Le modèle sauvegardé, entraîné sur 2012–2020, est utilisé **sans réentraînement ni modification des prétraitements**.

L'historique est figé fin 2020, y compris pour les baselines ; les résultats de validation et de test n'y entrent pas. La constante est la moyenne du train. Les dates test sont celles déjà inscrites dans le protocole. L'évaluation reste rétrospective, sur la géographie 2026.

Cette étape ouvre le test : les décisions ultérieures fondées sur ses scores ne pourront plus être présentées comme indépendantes de ce test. Les cellules des étapes précédentes conservent les états et résultats historiques de l'expérience.

In [16]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
from pathlib import Path
import sys
import hashlib
import json
from datetime import datetime, timezone
import joblib
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "environment.yml").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from scripts.integration.build_modeling_dataset import clean_outcomes

final_dir = ROOT / "reports/generated/hist_gradient_boosting/final_test"
final_dir.mkdir(parents=True, exist_ok=True)
selected_path = ROOT / "models/hist_gradient_boosting/selected_regularization.joblib"
selected_hash = hashlib.sha256(selected_path.read_bytes()).hexdigest()
selected = joblib.load(selected_path)
frozen_manifest = selected["manifest"]
assert frozen_manifest["selected_candidate"] == "very_compact"
assert frozen_manifest["refit_train_validation"] is False
expected_settings = {"max_leaf_nodes": 7, "min_samples_leaf": 200, "l2_regularization": 5.0,
                     "max_iter": 100, "learning_rate": 0.1, "early_stopping": False, "random_state": 42}
actual_settings = selected["pipeline"].named_steps["regressor"].get_params()
assert all(actual_settings[key] == value for key, value in expected_settings.items())
table_path = ROOT / "data/processed/modeling/municipal_structured.parquet"
ofs_path = ROOT / "data/processed/ofs-votes/votes_communes.parquet"
assert hashlib.sha256(table_path.read_bytes()).hexdigest() == frozen_manifest["dataset_sha256"]
assert hashlib.sha256(ofs_path.read_bytes()).hexdigest() == frozen_manifest["history_source_sha256"]
final_protocol = {
    "evaluation_started_at_utc": datetime.now(timezone.utc).isoformat(),
    "mode": "exploratory_retrospective",
    "choices_frozen": True,
    "selected_model_sha256": selected_hash,
    "dataset_sha256": frozen_manifest["dataset_sha256"],
    "history_source_sha256": frozen_manifest["history_source_sha256"],
    "settings": frozen_manifest["settings"],
    "train_end": frozen_manifest["train_end"],
    "test_dates": frozen_manifest["partition_dates"]["test"],
    "history_policy": frozen_manifest["history_policy"],
    "model_refitted": False,
    "selection": "settings selected on validation before opening test",
    "metric": "equal-row MAE in percentage points",
}
# Consigner les choix avant de calculer toute prédiction ou métrique test.
(final_dir / "frozen_protocol.json").write_text(json.dumps(final_protocol, ensure_ascii=False, indent=2), encoding="utf-8")

final_data = pd.read_parquet(table_path)
test_dates = pd.to_datetime(final_protocol["test_dates"])
test_data = final_data.loc[final_data.vote_date.isin(test_dates)].copy()
assert sorted(test_data.vote_date.dt.strftime("%Y-%m-%d").unique()) == sorted(final_protocol["test_dates"])
assert test_data.vote_date.min() > pd.Timestamp(frozen_manifest["validation_end"])
assert not test_data.duplicated(["municipality_id", "vote_id"]).any()
assert test_data.yes_share.notna().all() and test_data.yes_share.between(0, 1).all()
eligible_history, _ = clean_outcomes(pd.read_parquet(ofs_path))
history_pool = eligible_history.loc[eligible_history.vote_date.le(pd.Timestamp(frozen_manifest["train_end"]))]
frozen_history = history_pool.groupby("municipality_id").agg(
    history_mean_yes=("yes_share", "mean"), history_n_objects=("yes_share", "count"),
    history_last_vote_date=("vote_date", "max"),
)
for column in frozen_history.columns:
    test_data[column] = test_data.municipality_id.map(frozen_history[column])
test_data["history_n_objects"] = test_data.history_n_objects.fillna(0).astype("int64")
test_data["history_missing"] = test_data.history_mean_yes.isna()
assert test_data.history_last_vote_date.dropna().le(pd.Timestamp(frozen_manifest["train_end"])).all()
test_matrix = test_data[selected["features"]].copy()
for column in frozen_manifest["numeric_features"] + frozen_manifest["indicator_features"]:
    test_matrix[column] = test_matrix[column].astype(float)
for column in frozen_manifest["categorical_features"]:
    test_matrix[column] = test_matrix[column].astype(object).where(test_matrix[column].notna(), np.nan)
print("Test :", len(test_data), "lignes ;", test_data.vote_id.nunique(), "objets ;", test_data.vote_date.nunique(), "dates.")

Test : 37890 lignes ; 18 objets ; 7 dates.


In [17]:
raw_test_prediction = selected["pipeline"].predict(test_matrix)
assert np.isfinite(raw_test_prediction).all()
test_predictions = test_data[["municipality_id", "vote_id", "vote_date", "yes_share"]].copy()
test_predictions["constant"] = frozen_manifest["constant_prediction"]
test_predictions["municipal_history"] = test_data.history_mean_yes.fillna(frozen_manifest["constant_prediction"])
test_predictions["hist_gradient_boosting"] = np.clip(raw_test_prediction, 0, 1)
test_score_rows, object_frames, date_frames = [], [], []
for name in ["constant", "municipal_history", "hist_gradient_boosting"]:
    errors = test_predictions[["vote_id", "vote_date"]].copy()
    errors["error_pp"] = (test_predictions[name] - test_predictions.yes_share) * 100
    errors["absolute_error_pp"] = errors.error_pp.abs()
    assert errors.absolute_error_pp.notna().all()
    by_object = errors.groupby(["vote_id", "vote_date"], as_index=False).agg(
        mae_pp=("absolute_error_pp", "mean"), bias_pp=("error_pp", "mean"), rows=("error_pp", "size"),
    )
    by_date = errors.groupby("vote_date", as_index=False).absolute_error_pp.mean()
    object_frames.append(by_object.assign(model=name))
    date_frames.append(by_date.assign(model=name))
    test_score_rows.append({"model": name, "mae_pp": float(errors.absolute_error_pp.mean()),
        "mean_object_mae_pp": float(by_object.mae_pp.mean()),
        "mean_date_mae_pp": float(by_date.absolute_error_pp.mean()),
        "bias_pp": float(errors.error_pp.mean())})
test_metrics = pd.DataFrame(test_score_rows).set_index("model")
test_metrics["gain_vs_history_pp"] = test_metrics.loc["municipal_history", "mae_pp"] - test_metrics.mae_pp
display(test_metrics.round(3))
test_objects = pd.concat(object_frames, ignore_index=True)
test_by_date = pd.concat(date_frames, ignore_index=True)
object_comparison = test_objects.pivot(index=["vote_id", "vote_date"], columns="model", values="mae_pp").reset_index()
object_comparison["gain_vs_history_pp"] = object_comparison.municipal_history - object_comparison.hist_gradient_boosting
object_comparison["vote_id"] = object_comparison.vote_id.astype("string")
titles = pd.read_parquet(ROOT / "data/processed/swissvotes/swissvotes_clean.parquet")[["vote_key", "titel_kurz_f"]]
titles = titles.rename(columns={"vote_key": "vote_id", "titel_kurz_f": "title"})
titles["vote_id"] = titles.vote_id.astype("string")
object_comparison = object_comparison.merge(titles, on="vote_id", how="left", validate="many_to_one")
display(object_comparison.sort_values("hist_gradient_boosting", ascending=False).round(3))
date_comparison = test_by_date.pivot(index="vote_date", columns="model", values="absolute_error_pp")
date_comparison["gain_vs_history_pp"] = date_comparison.municipal_history - date_comparison.hist_gradient_boosting
display(date_comparison.round(3))

<ipython-input-1-7bb3d8c11ed8>:35: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  display(object_comparison.sort_values("hist_gradient_boosting", ascending=False).round(3))


,mae_pp,mean_object_mae_pp,mean_date_mae_pp,bias_pp,gain_vs_history_pp
model,,,,,
constant,14.664,14.664,15.744,-0.555,-0.044
municipal_history,14.620,14.620,15.889,1.160,0.000
hist_gradient_boosting,13.385,13.385,14.248,3.107,1.235


,vote_id,vote_date,constant,hist_gradient_boosting,municipal_history,gain_vs_history_pp,title
3,6650,2024-03-03,14.388,26.324,12.764,-13.56,Initiative pour une 13e rente AVS
7,6690,2024-06-09,16.967,24.919,18.793,-6.127,Initiative «Pour la liberté et l’intégrité phy...
10,6720,2024-09-22,14.503,21.137,16.197,-4.94,Réforme de la prévoyance professionnelle
9,6710,2024-09-22,17.325,18.39,18.763,0.373,Initiative biodiversité
15,6770,2025-02-09,21.808,16.088,23.439,7.351,Initiative «pour la responsabilité environneme...
0,6620,2023-06-18,31.951,14.826,30.236,15.41,Imposition minimale de l'OCDE
16,6780,2025-09-28,16.687,13.711,16.421,2.71,Impôts cantonaux sur les résidences secondaires
14,6760,2024-11-24,11.031,13.054,10.948,-2.106,Financement uniforme des prestations ambulatoi...
8,6700,2024-06-09,17.757,12.526,15.887,3.361,Loi sur les énergies renouvelables (acte modif...
5,6670,2024-06-09,11.841,10.545,11.563,1.017,Initiative d’allègement des primes


model,constant,hist_gradient_boosting,municipal_history,gain_vs_history_pp
vote_date,,,,
2023-06-18,17.846,10.478,15.881,5.403
2024-03-03,18.839,18.016,18.885,0.869
2024-06-09,14.474,14.01,14.685,0.675
2024-09-22,15.914,19.763,17.48,-2.283
2024-11-24,9.277,9.526,9.455,-0.07
2025-02-09,21.808,16.088,23.439,7.351
2025-09-28,12.047,11.852,11.394,-0.458


In [18]:
test_predictions.to_parquet(final_dir / "test_predictions.parquet", index=False)
test_metrics.to_csv(final_dir / "test_metrics.csv")
test_objects.to_csv(final_dir / "test_by_object.csv", index=False)
object_comparison.to_csv(final_dir / "test_object_comparison.csv", index=False)
date_comparison.to_csv(final_dir / "test_by_date.csv")
test_summary = {
    **final_protocol,
    "test_evaluated": True,
    "test_rows": len(test_data),
    "test_objects": int(test_data.vote_id.nunique()),
    "test_dates_count": int(test_data.vote_date.nunique()),
    "history_fallback_rows": int(test_data.history_missing.sum()),
    "clipped_predictions": int(((raw_test_prediction < 0) | (raw_test_prediction > 1)).sum()),
    "objects_improved_vs_history": int(object_comparison.gain_vs_history_pp.gt(0).sum()),
    "dates_improved_vs_history": int(date_comparison.gain_vs_history_pp.gt(0).sum()),
    "metrics": test_metrics.to_dict(orient="index"),
    "validation_mae_pp": frozen_manifest["validation_mae_pp"],
}
assert hashlib.sha256(selected_path.read_bytes()).hexdigest() == selected_hash
(final_dir / "test_summary.json").write_text(json.dumps(test_summary, ensure_ascii=False, indent=2), encoding="utf-8")
print("Objets améliorés :", test_summary["objects_improved_vs_history"], "/", test_summary["test_objects"])
print("Dates améliorées :", test_summary["dates_improved_vs_history"], "/", test_summary["test_dates_count"])
print("Replis historiques :", test_summary["history_fallback_rows"])
print("Modèle sauvegardé inchangé. Test désormais évalué ; aucune adaptation aux résultats test.")

Objets améliorés : 11 / 18
Dates améliorées : 4 / 7
Replis historiques : 18
Modèle sauvegardé inchangé. Test désormais évalué ; aucune adaptation aux résultats test.


### Conclusion finale de cette expérience

Test 2023–2025 : **37,890 lignes, 18 objets et 7 dates**. Le modèle reste celui ajusté sur 2012–2020, sans réentraînement.

| Modèle | MAE test (points de pourcentage) |
|---|---:|
| Constante | 14.664 |
| Historique communal | 14.620 |
| Boosting sélectionné | **13.385** |

Le gain sur l'historique est de **1.235 point**, soit **8.4 %** de réduction de MAE. Le boosting gagne sur **11 objets sur 18** et **4 dates sur 7**. Sa MAE passe de **10.709** en validation à **13.385** sur le test ; ces périodes contiennent des objets différents et ne sont pas de difficulté identique.

La plus forte erreur concerne l'initiative pour une 13e rente AVS : **26,32 points de MAE**, contre **12,76** pour l'historique. Le biais moyen du boosting sur le test est de **+3,11 points** (surestimation du oui).

**Conclusion :** l'amélioration moyenne sur les baselines se retrouve sur ce test, mais reste modeste et irrégulière selon les objets. Une erreur moyenne de 13,38 points ne justifie pas de présenter le modèle comme une prévision précise. Il s'agit toujours d'une évaluation rétrospective, avec des limites de disponibilité historique des sources.
